In [0]:
from pyspark.sql.functions import expr, current_timestamp
import unicodedata

In [0]:
raw_path = '/Volumes/ct-esteira-dados-aviacao/raw/tarifas-dados'

In [0]:
df_tarifas = spark.read.format('csv')\
    .option('header', 'true')\
    .option('delimiter', ';')\
    .csv(raw_path)

In [0]:
linhas = df_tarifas.count()
linhas

In [0]:
def clean_col(col_name):
    col_name = col_name.strip().lower().replace(" ", '_')

    col_name = ''.join(
        c for c in unicodedata.normalize("NFKD", col_name)
        if not unicodedata.combining(c)
    )
    return col_name

df_tarifas = df_tarifas.toDF(
    *[clean_col(c) for c in df_tarifas.columns]
    )

In [0]:
display(df_tarifas)

In [0]:
df_tarifas = df_tarifas.withColumn('load_data', current_timestamp())

In [0]:
df_tarifas.write \
    .format('delta')\
    .option('overwriteSchema', 'true')\
    .mode('overwrite')\
    .partitionBy('empresa') \
    .saveAsTable("`ct-esteira-dados-aviacao`.raw.tarifas_voos")
